# Readers

Notebook was prepared as a sample tool to read data from different data sources. Moreover notebook includes some simple exploratory data analysis for data overview purposes.

In [2]:
# Reload modules to reflect changes made in external scripts
%load_ext autoreload
%autoreload 2

In [3]:
import os
import sys
from pathlib import Path
from pprint import pprint as print

import requests
import pandas as pd
import numpy as np


# Setup pitch and plot
from mplsoccer import Pitch, VerticalPitch


In [4]:
CURRENT_WORKING_DIR = Path(os.getcwd())
WORKING_DIR = CURRENT_WORKING_DIR.parent
DATA_DIR = WORKING_DIR / "data"

sys.path.append(str(WORKING_DIR))

## SkillCorner

Data loading is based on the SkillCorner tutorial guide. More information under below link :arrow_down:

**Link:** https://github.com/SkillCorner/opendata

In [5]:
# Content URL for the SkillCorner opendata repository
content_url = "https://api.github.com/repos/SkillCorner/opendata/contents/data/matches"

# Get the list of available matches
response = requests.get(content_url)
matches = response.json()

# Display available match IDs
match_ids = [match['name'] for match in matches if match['type'] == 'dir']
print(f"Available matches: {len(match_ids)}")
print(match_ids)

'Available matches: 10'
['1886347',
 '1899585',
 '1925299',
 '1953632',
 '1996435',
 '2006229',
 '2011166',
 '2013725',
 '2015213',
 '2017461']


## Data Readers ###

In [21]:
content_tracking_data

,name,path,sha,size,url,html_url,git_url,download_url,type,content,encoding,_links
0,1886347_tracking_extrapolated.jsonl,data/matches/1886347/1886347_tracking_extrapol...,6c1d01ae3bafa28cf94e4b7a581fb3779effe4bd,89280839,https://api.github.com/repos/SkillCorner/opend...,https://github.com/SkillCorner/opendata/blob/m...,https://api.github.com/repos/SkillCorner/opend...,https://media.githubusercontent.com/media/Skil...,file,dmVyc2lvbiBodHRwczovL2dpdC1sZnMuZ2l0aHViLmNvbS...,base64,{'self': 'https://api.github.com/repos/SkillCo...


In [6]:
# Example: Load tracking data for a specific match
match_id = 1886347

# Construct the raw tracking content URL
content_tracking_url = content_url + f"/{match_id}/{match_id}_tracking_extrapolated.jsonl"  # Data is stored using GitLFS
content_tracking_data = pd.read_json(content_tracking_url, lines=True)
print(f"Downloaded tracking data URL: {content_tracking_data['download_url'][0]}")

# Construct the raw match content URL
content_match_url = content_url + f"/{match_id}/{match_id}_match.json"  # Data is stored using GitLFS
content_match_data = pd.read_json(content_match_url, lines=True)
print(f"Downloaded match data URL: {content_match_data['download_url'][0]}")

('Downloaded tracking data URL: '
 'https://media.githubusercontent.com/media/SkillCorner/opendata/master/data/matches/1886347/1886347_tracking_extrapolated.jsonl')
('Downloaded match data URL: '
 'https://raw.githubusercontent.com/SkillCorner/opendata/master/data/matches/1886347/1886347_match.json')


## Player data

Extract player and ball data. Preprocess raw data frame into processed, ready for analytics data frame. 

In [ ]:
# Example 1: Preprocess tracking data
from football_ml.jobs.data_gathering_job import preprocess_tracking_data

# Read the JSON tracking data as a JSON object
raw_tracking_data = pd.read_json(content_tracking_data['download_url'][0], lines=True)



,x,y,player_id,is_detected,frame,timestamp,period,possession_player_id,possession_group,ball_x,ball_y,ball_z,is_detected_ball,match_id
0,-39.63,-0.08,51009,False,10,2025-12-30,1.0,NaN,None,0.32,0.38,0.13,True,1886347
1,-19.21,-9.18,176224,True,10,2025-12-30,1.0,NaN,None,0.32,0.38,0.13,True,1886347
2,-21.83,0.47,51649,True,10,2025-12-30,1.0,NaN,None,0.32,0.38,0.13,True,1886347
3,-1.16,-32.47,50983,True,10,2025-12-30,1.0,NaN,None,0.32,0.38,0.13,True,1886347
4,-18.88,15.73,735578,True,10,2025-12-30,1.0,NaN,None,0.32,0.38,0.13,True,1886347


In [87]:
# Expand the ball_data column
ball_expanded = pd.json_normalize(raw_tracking_data['ball_data'])
possession_expanded = pd.json_normalize(raw_tracking_data['possession'])

# Combine with original dataframe
ball_tracking_df = pd.concat([
    raw_tracking_data.drop(columns=["ball_data", "player_data", "image_corners_projection", "possession", "frame"], axis=1),
    ball_expanded.rename(columns={"x": "ball_x", "y": "ball_y", "z": "ball_z"}),
    possession_expanded.rename(columns={"player_id": "possession_player_id", "group": "possession_team_group"}),    
    ], 
axis=1).drop_duplicates().dropna(subset=["ball_x", "ball_y", "ball_z"])
ball_tracking_df["match_id"] = match_id
ball_tracking_df = ball_tracking_df.reset_index(drop=True).reset_index().rename(columns={"index": "ball_tracking_id"})
columns_ = [
    "ball_tracking_id", "timestamp",
    "ball_x", "ball_y", "ball_z",
    "is_detected", "period", "match_id", "possession_player_id", "possession_team_group"
]
ball_tracking_df = ball_tracking_df[columns_]
print(f"Ball tracking data shape: {ball_tracking_df.shape}")
ball_tracking_df.head()

'Ball tracking data shape: (43458, 10)'


,ball_tracking_id,timestamp,ball_x,ball_y,ball_z,is_detected,period,match_id,possession_player_id,possession_team_group
0,0,2025-12-30 00:00:00.000,0.32,0.38,0.13,True,1.0,1886347,NaN,None
1,1,2025-12-30 00:00:00.100,0.54,0.08,0.22,True,1.0,1886347,NaN,None
2,2,2025-12-30 00:00:00.200,0.57,-0.07,0.19,True,1.0,1886347,NaN,None
3,3,2025-12-30 00:00:00.300,0.56,-0.07,0.14,True,1.0,1886347,NaN,None
4,4,2025-12-30 00:00:00.400,0.59,-0.03,0.14,True,1.0,1886347,NaN,None


In [88]:
# Process the raw data to get players tracking data
players_tracking_df = pd.json_normalize(
    raw_tracking_data.to_dict("records"),
    "player_data",
    ["frame", "timestamp", "period"],
).reset_index().rename(columns={"x": "player_x", "y": "player_y", "index": "player_tracking_id"})
players_tracking_df["match_id"] = match_id
players_tracking_df.head()

,player_tracking_id,player_x,player_y,player_id,is_detected,frame,timestamp,period,match_id
0,0,-39.63,-0.08,51009,False,10,2025-12-30,1.0,1886347
1,1,-19.21,-9.18,176224,True,10,2025-12-30,1.0,1886347
2,2,-21.83,0.47,51649,True,10,2025-12-30,1.0,1886347
3,3,-1.16,-32.47,50983,True,10,2025-12-30,1.0,1886347
4,4,-18.88,15.73,735578,True,10,2025-12-30,1.0,1886347


In [ ]:
# Process the raw data
processed_tracking_df = preprocess_tracking_data(raw_tracking_data)
processed_tracking_df["match_id"] = match_id
processed_tracking_df.head()

In [12]:
path = DATA_DIR / "raw" / "all_matches_tracking.parquet"
all_matches_tracking_df = pd.read_parquet(path)

## Match data

In [8]:
# Exampke 2: Preprocess match and players data

from football_ml.jobs.data_gathering_job import preprocess_players_data

# Read the JSON match data as a JSON object
response = requests.get(content_match_data['download_url'][0])
raw_match_data = response.json()
raw_match_df = pd.json_normalize(raw_match_data, max_level=2)
processed_players_df = preprocess_players_data(raw_match_df)
processed_players_df.head()

,start_time,end_time,match_id,match_name,date_time,home_team.name,away_team.name,player_id,short_name,number,team_id,team_name,player_role.position_group,total_time,player_role.name,player_role.acronym,is_gk,direction_player_1st_half,direction_player_2nd_half
0,00:00:00,01:25:21,1886347,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,38673,G. May,10,4177,Auckland FC,Center Forward,5121,Center Forward,CF,False,right_to_left,left_to_right
1,00:00:00,None,1886347,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,51713,C. Elliott,17,4177,Auckland FC,Full Back,5400,Right Back,RB,False,right_to_left,left_to_right
2,00:00:00,01:16:37,1886347,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,50951,J. Brimmer,22,4177,Auckland FC,Center Forward,4597,Center Forward,CF,False,right_to_left,left_to_right
3,00:00:00,01:24:58,1886347,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,50978,C. Timmins,19,1805,Newcastle United Jets FC,Midfield,5098,Left Defensive Midfield,LDM,False,left_to_right,right_to_left
4,00:00:00,None,1886347,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,133498,F. De Vries,15,4177,Auckland FC,Full Back,5400,Left Back,LB,False,right_to_left,left_to_right


In [11]:
path = DATA_DIR / "raw" / "all_matches_players.parquet"
all_matches_players_df = pd.read_parquet(path)

In [17]:
all_matches_tracking_df.possession_player_id.unique()

array([    nan, 966120.,  51649.,  23418.,  33697., 133498.,  50983.,
       965685.,  51009., 735578.,  51713.,  14736.,  50978., 176224.,
       735573., 795505., 735574.,  50951.,  38673.,  51667., 285188.,
       795507., 133501., 560992., 795506.,  43829., 163972., 797297.,
       800320.,  31147.,  23909.,  42510.,  51015.,  51028., 698137.,
        34718.,  51017.,   4322.,  27003., 799092.,  51046.,  26969.,
       133495., 965684.,   6799., 159938., 799091.,  51002., 797298.,
       965697.,  51050., 560983., 560986.,  11897., 795530., 560984.,
       809166., 560882.,  51694.,  50982., 966114., 560989., 560985.,
       287934.,  51043., 966112., 560987.,  50999.,  69111., 795537.,
       795533., 966115.,  51722., 560988., 965698., 795541., 333110.,
       809869., 966113.,  51681.,   9520.,  50992.,  34423., 771859.,
        11891.,  51668., 104563., 771861.,  29856., 795511., 795510.,
        23930.,  51051.,  51039.,  22937., 923862.,  10549., 771857.,
       795151., 7955

In [21]:
mask = (all_matches_tracking_df['player_id'] == 966120) & (all_matches_tracking_df['possession_player_id'] == 966120) & (all_matches_tracking_df['match_id'] == 1886347)
all_matches_tracking_df[mask].sort_values(by=['timestamp'])

,x,y,player_id,is_detected,frame,timestamp,period,possession_player_id,possession_group,ball_x,ball_y,ball_z,is_detected_ball,match_id
406,0.73,0.49,966120,True,28,2025-12-29 00:00:01.800,1.0,966120.0,away team,0.03,-0.36,0.33,True,1886347
2100,21.72,14.87,966120,True,105,2025-12-29 00:00:09.500,1.0,966120.0,away team,20.87,16.98,1.92,True,1886347
2122,21.94,14.99,966120,True,106,2025-12-29 00:00:09.600,1.0,966120.0,away team,21.73,16.53,1.59,True,1886347
2144,22.14,15.07,966120,True,107,2025-12-29 00:00:09.700,1.0,966120.0,away team,22.31,16.23,1.19,True,1886347
2166,22.32,15.12,966120,True,108,2025-12-29 00:00:09.800,1.0,966120.0,away team,22.72,15.99,0.82,True,1886347
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
669327,12.55,0.98,966120,True,36620,2025-12-29 00:59:42.000,2.0,966120.0,away team,13.30,1.13,0.78,True,1886347
669349,12.67,0.85,966120,True,36621,2025-12-29 00:59:42.100,2.0,966120.0,away team,13.28,0.98,0.54,True,1886347
669371,12.79,0.74,966120,True,36622,2025-12-29 00:59:42.200,2.0,966120.0,away team,13.25,0.77,0.43,True,1886347
669393,12.91,0.63,966120,True,36623,2025-12-29 00:59:42.300,2.0,966120.0,away team,13.33,0.61,0.40,True,1886347


In [ ]:
# Player table

# Match table

In [ ]:
# Match table columns -> dim_matches
[
    "match_id",
    "match_name",
    "date_time",
    "start_time",
    "end_time",
    "total_time",
    "home_team.name",
    "away_team.name",
]

# Player table columns -> dim_players
[
    "player_id",
    "short_name",  # player_name
    "number",  # player_jersey_number
    "team_id",  # player_team_id
]

# Team table columns - dim_teams
[
    "team_id",
    "team_name",
]

# Event table columns -> fact_events
[
    "event_id",
    "match_id",
    "event_type",
    "event_subtype",
    "period",
    "timestamp",
    "team_id",
    "player_id",
    "position.x",
    "position.y",
    "outcome",
    "target_position.x",
    "target_position.y",
    "body_part",
    "assist_player_id",
    "shot_outcome",
    "shot_xg",
]

# Events
[
    "event_id",  # new id based on match id and player id
    "match_id",
    "player_id",
    "player_role.position_group",
    "player_role.name",
    "player_role.acronym",
    "is_gk",
    "direction_player_1st_half",
    "direction_player_2nd_half",
]

In [22]:
all_matches_players_df.head()

,start_time,end_time,match_id,match_name,date_time,home_team.name,away_team.name,player_id,short_name,number,team_id,team_name,player_role.position_group,total_time,player_role.name,player_role.acronym,is_gk,direction_player_1st_half,direction_player_2nd_half
0,00:00:00,01:25:21,1886347,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,38673,G. May,10,4177,Auckland FC,Center Forward,5121,Center Forward,CF,False,right_to_left,left_to_right
1,00:00:00,None,1886347,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,51713,C. Elliott,17,4177,Auckland FC,Full Back,5400,Right Back,RB,False,right_to_left,left_to_right
2,00:00:00,01:16:37,1886347,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,50951,J. Brimmer,22,4177,Auckland FC,Center Forward,4597,Center Forward,CF,False,right_to_left,left_to_right
3,00:00:00,01:24:58,1886347,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,50978,C. Timmins,19,1805,Newcastle United Jets FC,Midfield,5098,Left Defensive Midfield,LDM,False,left_to_right,right_to_left
4,00:00:00,None,1886347,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,133498,F. De Vries,15,4177,Auckland FC,Full Back,5400,Left Back,LB,False,right_to_left,left_to_right


In [15]:
all_matches_tracking_df.head()

,x,y,player_id,is_detected,frame,timestamp,period,possession_player_id,possession_group,ball_x,ball_y,ball_z,is_detected_ball,match_id
0,-39.63,-0.08,51009,False,10,2025-12-29,1.0,NaN,None,0.32,0.38,0.13,True,1886347
1,-19.21,-9.18,176224,True,10,2025-12-29,1.0,NaN,None,0.32,0.38,0.13,True,1886347
2,-21.83,0.47,51649,True,10,2025-12-29,1.0,NaN,None,0.32,0.38,0.13,True,1886347
3,-1.16,-32.47,50983,True,10,2025-12-29,1.0,NaN,None,0.32,0.38,0.13,True,1886347
4,-18.88,15.73,735578,True,10,2025-12-29,1.0,NaN,None,0.32,0.38,0.13,True,1886347


In [14]:
all_matches_players_df

,start_time,end_time,match_id,match_name,date_time,home_team.name,away_team.name,player_id,short_name,number,team_id,team_name,player_role.position_group,total_time,player_role.name,player_role.acronym,is_gk,direction_player_1st_half,direction_player_2nd_half
0,00:00:00,01:25:21,1886347,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,38673,G. May,10,4177,Auckland FC,Center Forward,5121,Center Forward,CF,False,right_to_left,left_to_right
1,00:00:00,None,1886347,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,51713,C. Elliott,17,4177,Auckland FC,Full Back,5400,Right Back,RB,False,right_to_left,left_to_right
2,00:00:00,01:16:37,1886347,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,50951,J. Brimmer,22,4177,Auckland FC,Center Forward,4597,Center Forward,CF,False,right_to_left,left_to_right
3,00:00:00,01:24:58,1886347,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,50978,C. Timmins,19,1805,Newcastle United Jets FC,Midfield,5098,Left Defensive Midfield,LDM,False,left_to_right,right_to_left
4,00:00:00,None,1886347,Auckland FC vs Newcastle United Jets FC,2024-11-30T04:00:00Z,Auckland FC,Newcastle United Jets FC,133498,F. De Vries,15,4177,Auckland FC,Full Back,5400,Left Back,LB,False,right_to_left,left_to_right
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304,00:00:00,None,2017461,Melbourne Victory Football Club vs Auckland FC,2025-05-17T09:35:00Z,Melbourne Victory Football Club,Auckland FC,800322,J. Inserra,16,868,Melbourne Victory Football Club,Full Back,5400,Right Back,RB,False,left_to_right,right_to_left
305,00:00:00,None,2017461,Melbourne Victory Football Club vs Auckland FC,2025-05-17T09:35:00Z,Melbourne Victory Football Club,Auckland FC,51678,J. Duncan,25,868,Melbourne Victory Football Club,Other,5400,Goalkeeper,GK,True,left_to_right,right_to_left
306,00:00:00,01:09:04,2017461,Melbourne Victory Football Club vs Auckland FC,2025-05-17T09:35:00Z,Melbourne Victory Football Club,Auckland FC,29075,N. Vergos,9,868,Melbourne Victory Football Club,Center Forward,4144,Center Forward,CF,False,left_to_right,right_to_left
307,00:00:00,01:09:04,2017461,Melbourne Victory Football Club vs Auckland FC,2025-05-17T09:35:00Z,Melbourne Victory Football Club,Auckland FC,11885,D. Arzani,7,868,Melbourne Victory Football Club,Wide Attacker,4144,Right Winger,RW,False,left_to_right,right_to_left


## Merging data frame

In [13]:
merged_df = all_matches_tracking_df.merge(
    all_matches_players_df, on=["player_id", "match_id"], how="left"
)
merged_df["time_in_minutes"] = np.round(merged_df["timestamp"].dt.hour * 60 + merged_df["timestamp"].dt.minute + merged_df["timestamp"].dt.second / 60, 2)
merged_df.head()

,x,y,player_id,is_detected,frame,timestamp,period,possession_player_id,possession_group,ball_x,...,team_id,team_name,player_role.position_group,total_time,player_role.name,player_role.acronym,is_gk,direction_player_1st_half,direction_player_2nd_half,time_in_minutes
0,-39.63,-0.08,51009,False,10,2025-12-29,1.0,NaN,None,0.32,...,1805,Newcastle United Jets FC,Other,5400,Goalkeeper,GK,True,left_to_right,right_to_left,0.0
1,-19.21,-9.18,176224,True,10,2025-12-29,1.0,NaN,None,0.32,...,1805,Newcastle United Jets FC,Central Defender,5400,Right Center Back,RCB,False,left_to_right,right_to_left,0.0
2,-21.83,0.47,51649,True,10,2025-12-29,1.0,NaN,None,0.32,...,1805,Newcastle United Jets FC,Central Defender,5400,Left Center Back,LCB,False,left_to_right,right_to_left,0.0
3,-1.16,-32.47,50983,True,10,2025-12-29,1.0,NaN,None,0.32,...,1805,Newcastle United Jets FC,Full Back,5400,Right Back,RB,False,left_to_right,right_to_left,0.0
4,-18.88,15.73,735578,True,10,2025-12-29,1.0,NaN,None,0.32,...,1805,Newcastle United Jets FC,Full Back,5400,Left Back,LB,False,left_to_right,right_to_left,0.0


In [ ]:
print(f"Processed tracking data shape: {all_matches_tracking_df.shape}")
print(f"Processed players data shape: {all_matches_players_df.shape}")
print(f"Merged data shape: {merged_df.shape}")

**Analysis ideas:**
 - free kicks, corners - players movements
 - Midfielders - off ball movements, number of offensive / defensive passes, passes based on the number of opponent players
 - Center Backs - off ball movements in offensive part of the game, passes
 - Before goal team movement

**ML model ideas:**
- Clustering - players similarities
    - off ball movements

In [ ]:
# Filtering for frames with a team in possession
filtered_df = merged_df[
    merged_df["possession_group"].notnull()
].copy()


# We basically want to convert the X and Y to make sure we're always visualizing left to right. A player's XY will depend on the half so the straight average doesn't work

filtered_df["direction_player"] = np.where(
    filtered_df["period"] == 1,
    filtered_df["direction_player_1st_half"],
    filtered_df["direction_player_2nd_half"],
)
filtered_df["x"] = np.where(
    filtered_df["direction_player"] == "right_to_left",
    -filtered_df["x"],
    filtered_df["x"],
)  # Convert X
filtered_df["y"] = np.where(
    filtered_df["direction_player"] == "right_to_left",
    -filtered_df["y"],
    filtered_df["y"],
)  # Convert Y

# Create some flags in case we need them later
filtered_df["possession_team_name"] = np.where(
    filtered_df["possession_group"] == "home team",
    filtered_df["home_team.name"],
    filtered_df["away_team.name"],
)
filtered_df["possession_flag"] = np.where(
    filtered_df["possession_team_name"] == filtered_df["team_name"], "IP", "OOP"
)


# At this point all of our players have their X and Y adjusted from left to right , so we can aggregate for across the game

aggregated_df = (
    filtered_df.groupby(
        [
            "player_id",
            "possession_group",
            "team_name",
            "possession_team_name",
            "possession_flag",  # This will allow us to check the position of the player when the team is in or out of possession
            "start_time",  # With this we can filter in or out the players that came on as subs
            "number",
            "is_gk",
        ]
    )[["x", "y"]]
    .mean()
    .reset_index()
)

aggregated_df.sort_values(["possession_flag", "team_name"])

# Visualization example

In [ ]:
mask = (merged_df['player_id'] == 23909) & (merged_df['match_id'] == 1899585) & ()
viz_df = merged_df[mask]

In [ ]:
merged_df.columns

In [ ]:
!pip install nbformat==4.3.0

In [ ]:
from plotly import graph_objects as go

mask = viz_df["period"] == 1


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=viz_df.loc[mask, 'time_in_minutes'],
    y=viz_df.loc[mask, 'x'],
    mode='lines+markers',
    name='x'
))
fig.add_trace(go.Scatter(
    x=viz_df.loc[mask, 'time_in_minutes'],
    y=viz_df.loc[mask, 'ball_x'],
    mode='lines+markers',
    name='ball_x'
))
fig.update_layout(
    title="Player X Position and Ball X Position Over Time (Period 1)",
    xaxis_title="Time (minutes)",
    yaxis_title="X Position (meters)",
    legend_title="Legend"
)
fig.show()

In [ ]:

# plt.scatter(viz_df.loc[mask, 'time_in_minutes'], viz_df.loc[mask, 'x'], s=10, marker='.', label='x')
plt.plot(viz_df.loc[mask, 'time_in_minutes'], viz_df.loc[mask, 'x'], label='x')
# plt.scatter(viz_df.loc[mask, 'time_in_minutes'], viz_df.loc[mask, 'ball_x'], s=10, marker='.', label='ball_x')
plt.plot(viz_df.loc[mask, 'time_in_minutes'], viz_df.loc[mask, 'ball_x'], label='ball_x')
plt.legend()

In [ ]:
# Teams
aggregated_df.team_name.unique().tolist()

In [ ]:
possession = "IP"  # 'IP' or 'OOP'
team = 'Brisbane Roar FC'  # Pick one team, you can use the name directly

In [ ]:
pitch = Pitch(
    pitch_type="skillcorner",
    line_alpha=0.75,
    pitch_length=105,
    pitch_width=68,
    pitch_color="#001400",
    line_color="white",
    linewidth=1.5,
)
fig, ax = pitch.grid(figheight=8, endnote_height=0, title_height=0)

viz_ip = aggregated_df[
    (aggregated_df["possession_flag"] == possession)
    & (aggregated_df["team_name"] == team)
].reset_index(drop=True)

ax.scatter(
    viz_ip["x"],
    viz_ip["y"],
    c="#32FE6B",
    alpha=0.95,
    s=600,
    edgecolors="white",
    linewidths=2.5,
    zorder=10,
    label="team",
)

# Annotate player numbers
for i, row in viz_ip.iterrows():
    ax.text(
        row["x"],
        row["y"],
        str(row["number"]),
        color="black",
        fontweight="bold",
        fontsize=10,
        ha="center",
        va="center",
        zorder=16,
    )

ax.set_title(f"{team} Average Positions in Possession")

In [ ]:
aggregated_df["start_time"].unique()

In [ ]:
pitch = Pitch(
    pitch_type="skillcorner",
    line_alpha=0.75,
    pitch_length=105,
    pitch_width=68,
    pitch_color="#001400",
    line_color="white",
    linewidth=1.5,
)
fig, ax = pitch.grid(figheight=8, endnote_height=0, title_height=0)

viz_ip = aggregated_df[
    (aggregated_df["possession_flag"] == possession)
    & (aggregated_df["team_name"] == team)
    & (aggregated_df["start_time"] == "00:00:00")
].reset_index(drop=True)


ax.scatter(
    viz_ip["x"],
    viz_ip["y"],
    c="#32FE6B",
    alpha=0.95,
    s=600,
    edgecolors="white",
    linewidths=2.5,
    zorder=10,
    label="team",
)


# Annotate player numbers
for i, row in viz_ip.iterrows():
    ax.text(
        row["x"],
        row["y"],
        str(row["number"]),
        color="black",
        fontweight="bold",
        fontsize=10,
        ha="center",
        va="center",
        zorder=16,
    )

ax.set_title(f"{team} Average Positions in Possession")

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

import matplotlib.pyplot as plt

# Create pitch
pitch = Pitch(
    pitch_type="skillcorner",
    line_alpha=0.75,
    pitch_length=105,
    pitch_width=68,
    pitch_color="#001400",
    line_color="white",
    linewidth=1.5,
)

# Select a player to visualize
player_id = 23909  # You can change this to any player_id from the tracking data
player_name = merged_df[merged_df['player_id'] == player_id]['short_name'].values[0]
player_number = merged_df[merged_df['player_id'] == player_id]['number'].values[0]

# Filter tracking data for the selected player
player_tracking = merged_df[
    merged_df['player_id'] == player_id
].copy()

# Adjust coordinates based on direction (similar to what was done in filtered_df)
player_tracking["direction_player"] = np.where(
    player_tracking["period"] == 1,
    player_tracking["direction_player_1st_half"],
    player_tracking["direction_player_2nd_half"],
)
player_tracking["x_adjusted"] = np.where(
    player_tracking["direction_player"] == "right_to_left",
    -player_tracking["x"],
    player_tracking["x"],
)
player_tracking["y_adjusted"] = np.where(
    player_tracking["direction_player"] == "right_to_left",
    -player_tracking["y"],
    player_tracking["y"],
)

# Create figure
fig, ax = pitch.draw(figsize=(12, 8))

# Create heatmap using hexbin
heatmap = ax.hexbin(
    player_tracking['x_adjusted'],
    player_tracking['y_adjusted'],
    gridsize=20,
    cmap='hot',
    alpha=0.7,
    edgecolors='none',
    mincnt=1
)

# Add colorbar
cbar = plt.colorbar(heatmap, ax=ax)
cbar.set_label('Frequency', rotation=270, labelpad=20)

# Add title
plt.title(
    f"Heat Map: {player_name} (#{player_number}) - Full Match Movement",
    fontsize=14,
    fontweight='bold',
    color='white'
)

plt.tight_layout()
fig.show()